# 2.7 - Final Dataset Preparation

**Taller de Programación - UBA FCE | Grupo JLP**

---

## Objetivo

Preparar el dataset final limpio para modelado con la nueva base consolidada de sentiment (1979-2025):

1. **Cargar features completos** (step4 con clima)
2. **Mergear sentiment histórico completo** (GDELT 1.0 + 2.0: 1979-2025)
3. **Limpieza consolidada de NaNs** (evitar repetir en cada modelo)
4. **Validación de calidad** (verificar consistencia temporal)
5. **Guardar dataset listo para modelado** con metadata completa

**Salida:**
- `features_final_modeling.csv` - Dataset limpio sin NaNs con 46 años de sentiment
- `metadata_final_dataset.json` - Documentación completa de limpieza

## Setup

In [20]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import json
from datetime import datetime

# Agregar src al path
BASE_DIR = Path.cwd().parents[1]
sys.path.append(str(BASE_DIR / 'src'))

from config import PROCESSED_DIR, START_DATE, END_DATE, logger

# Configurar pandas display
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

print(f"✓ Base directory: {BASE_DIR}")
print(f"✓ Processed directory: {PROCESSED_DIR}")
print(f"✓ Período de análisis: {START_DATE} → {END_DATE}")

✓ Base directory: c:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal
✓ Processed directory: C:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal\data\processed
✓ Período de análisis: 2000-01-01 → 2025-12-07


## 1. Cargar Dataset Base + Todas las Features Académicas

**Datasets a mergear:**
1. `features_step4_climate.csv` - Base (3,186 features)
2. `cftc/cftc_features_2000_2025.csv` - CFTC (+11 features)
3. `sentiment/sentiment_features_1979_2025.csv` - **NUEVO:** Sentiment consolidado GDELT 1.0 + 2.0 (1979-2025, +10 features)
4. `bdi/bdi_features.csv` - Baltic Dry Index (+8 features)
5. `supply_demand/crop_conditions_all_features.csv` - Crop (+15 features)
6. `supply_demand/government_stocks_ers_all_features.csv` - Gov Stocks (+9 features)

**Total esperado:** 3,239 features
**Cobertura sentiment:** 46.9 años (16,753 días)

In [21]:
# Cargar dataset base (Step 4 - Climate)
df_base = pd.read_csv(PROCESSED_DIR / 'features_step4_climate.csv', index_col=0, parse_dates=True)

print("=" * 80)
print("DATASET BASE (Step 4 - Climate)")
print("=" * 80)
print(f"Shape: {df_base.shape}")
print(f"Período: {df_base.index.min()} → {df_base.index.max()}")
print(f"Features: {len(df_base.columns):,}")

# Cargar CFTC features
EXTERNAL_DIR = BASE_DIR / 'data' / 'external'
INTERIM_DIR = BASE_DIR / 'data' / 'interim'

cftc_path = EXTERNAL_DIR / 'cftc' / 'cftc_features_2000_2025.csv'
if cftc_path.exists():
    df_cftc = pd.read_csv(cftc_path, index_col=0, parse_dates=True)
    print(f"\n✓ CFTC: {df_cftc.shape} ({df_cftc.index.min()} → {df_cftc.index.max()})")
else:
    print(f"\n❌ CFTC no encontrado: {cftc_path}")
    df_cftc = None

# Cargar GDELT features consolidados (1979-2025)
gdelt_path = EXTERNAL_DIR / 'sentiment' / 'sentiment_features_1979_2025.csv'
if gdelt_path.exists():
    df_gdelt = pd.read_csv(gdelt_path, parse_dates=['date'])
    df_gdelt = df_gdelt.set_index('date')
    years_coverage = (df_gdelt.index.max() - df_gdelt.index.min()).days / 365.25
    print(f"✓ GDELT (CONSOLIDADO 1.0+2.0): {df_gdelt.shape}")
    print(f"  Período: {df_gdelt.index.min()} → {df_gdelt.index.max()}")
    print(f"  Cobertura: {years_coverage:.1f} años ({len(df_gdelt):,} días)")
else:
    print(f"❌ GDELT consolidado no encontrado: {gdelt_path}")
    df_gdelt = None

# Cargar BDI features
bdi_path = INTERIM_DIR / 'predictors' / 'bdi_features.csv'
if bdi_path.exists():
    df_bdi = pd.read_csv(bdi_path, index_col=0, parse_dates=True)
    print(f"✓ BDI: {df_bdi.shape} ({df_bdi.index.min()} → {df_bdi.index.max()})")
else:
    print(f"❌ BDI no encontrado: {bdi_path}")
    df_bdi = None

# Cargar Crop Conditions
crop_path = INTERIM_DIR / 'supply_demand' / 'crop_conditions_all_features.csv'
if crop_path.exists():
    df_crop = pd.read_csv(crop_path, index_col=0, parse_dates=True)
    print(f"✓ Crop Conditions: {df_crop.shape} ({df_crop.index.min()} → {df_crop.index.max()})")
else:
    print(f"❌ Crop Conditions no encontrado: {crop_path}")
    df_crop = None

# Cargar Government Stocks
stocks_path = INTERIM_DIR / 'supply_demand' / 'government_stocks_ers_all_features.csv'
if stocks_path.exists():
    df_stocks = pd.read_csv(stocks_path, index_col=0, parse_dates=True)
    print(f"✓ Gov Stocks: {df_stocks.shape} ({df_stocks.index.min()} → {df_stocks.index.max()})")
else:
    print(f"❌ Gov Stocks no encontrado: {stocks_path}")
    df_stocks = None

DATASET BASE (Step 4 - Climate)
Shape: (6731, 3186)
Período: 2000-01-03 00:00:00 → 2025-11-10 00:00:00
Features: 3,186

✓ CFTC: (28290, 18) (2000-01-04 00:00:00 → 2025-10-28 00:00:00)
✓ GDELT (CONSOLIDADO 1.0+2.0): (16753, 10)
  Período: 1979-01-01 00:00:00 → 2025-11-30 00:00:00
  Cobertura: 46.9 años (16,753 días)
✓ BDI: (6456, 8) (2000-01-04 00:00:00 → 2025-11-07 00:00:00)
✓ Crop Conditions: (337, 15) (2024-10-27 00:00:00 → 2025-09-28 00:00:00)
✓ Gov Stocks: (23834, 9) (1960-05-31 00:00:00 → 2025-08-31 00:00:00)


In [22]:
# Merge todas las features (left join para mantener todas las fechas del base)
df_features = df_base.copy()

# Merge CFTC
if df_cftc is not None:
    df_features = df_features.join(df_cftc, how='left', rsuffix='_cftc')
    print(f"\n✓ Merged CFTC: {df_features.shape}")

# Merge GDELT
if df_gdelt is not None:
    df_features = df_features.join(df_gdelt, how='left', rsuffix='_gdelt')
    print(f"✓ Merged GDELT: {df_features.shape}")

# Merge BDI
if df_bdi is not None:
    df_features = df_features.join(df_bdi, how='left', rsuffix='_bdi')
    print(f"✓ Merged BDI: {df_features.shape}")

# Merge Crop Conditions
if df_crop is not None:
    df_features = df_features.join(df_crop, how='left', rsuffix='_crop')
    print(f"✓ Merged Crop: {df_features.shape}")

# Merge Government Stocks
if df_stocks is not None:
    df_features = df_features.join(df_stocks, how='left', rsuffix='_stocks')
    print(f"✓ Merged Gov Stocks: {df_features.shape}")

print("\n" + "=" * 80)
print("DATASET COMPLETO CON TODAS LAS FEATURES")
print("=" * 80)
print(f"Shape final: {df_features.shape}")
print(f"Período: {df_features.index.min()} → {df_features.index.max()}")
print(f"Total features: {len(df_features.columns):,}")
print(f"\nFeatures esperadas: 3,239")
print(f"Features obtenidas: {len(df_features.columns):,}")


✓ Merged CFTC: (20173, 3204)
✓ Merged GDELT: (20173, 3214)
✓ Merged GDELT: (20173, 3214)
✓ Merged BDI: (20173, 3222)
✓ Merged BDI: (20173, 3222)
✓ Merged Crop: (20173, 3237)
✓ Merged Crop: (20173, 3237)
✓ Merged Gov Stocks: (20173, 3246)

DATASET COMPLETO CON TODAS LAS FEATURES
Shape final: (20173, 3246)
Período: 2000-01-03 00:00:00 → 2025-11-10 00:00:00
Total features: 3,246

Features esperadas: 3,239
Features obtenidas: 3,246
✓ Merged Gov Stocks: (20173, 3246)

DATASET COMPLETO CON TODAS LAS FEATURES
Shape final: (20173, 3246)
Período: 2000-01-03 00:00:00 → 2025-11-10 00:00:00
Total features: 3,246

Features esperadas: 3,239
Features obtenidas: 3,246


## 1.1 Merge Todas las Features con Left Join

## 2. Diagnóstico de Missing Values

Antes de limpiar, documentar estado inicial de NaNs por tipo de feature.

In [23]:
# Calcular missing values por columna
missing_summary = pd.DataFrame({
    'missing_count': df_features.isnull().sum(),
    'missing_pct': (df_features.isnull().sum() / len(df_features) * 100).round(2)
}).sort_values('missing_count', ascending=False)

missing_summary = missing_summary[missing_summary['missing_count'] > 0]

print("=" * 80)
print("MISSING VALUES POR FEATURE")
print("=" * 80)
print(f"\nTotal features con missing: {len(missing_summary)} / {len(df_features.columns)}")
print(f"Total missing values: {df_features.isnull().sum().sum():,}")
print(f"Porcentaje total: {(df_features.isnull().sum().sum() / df_features.size * 100):.2f}%\n")

if len(missing_summary) > 0:
    print("Top 20 features con más missing:")
    print(missing_summary.head(20))

MISSING VALUES POR FEATURE

Total features con missing: 2850 / 3246
Total missing values: 5,371,834
Porcentaje total: 8.20%

Top 20 features con más missing:
                                        missing_count  missing_pct
Baltic_Dry_Index_volume_price_to_ma7            20173        100.0
Heat_Stress_Days_price_to_ma30                  20173        100.0
Heat_Stress_Days_log_return30                   20173        100.0
Heat_Stress_Days_simple_return30                20173        100.0
Heat_Stress_Days_log_return90                   20173        100.0
Heat_Stress_Days_simple_return90                20173        100.0
Heat_Stress_Days_simple_return1                 20173        100.0
Heat_Stress_Days_log_return1                    20173        100.0
Heat_Stress_Days_log_return7                    20173        100.0
Heat_Stress_Days_simple_return7                 20173        100.0
Baltic_Dry_Index_volume_price_to_ma90           20173        100.0
Baltic_Dry_Index_volume_vol_ratio_7_30

In [24]:
# Clasificar missing por tipo de feature
def clasificar_feature(col_name):
    """Clasificar feature por su nombre para análisis de missing."""
    if '_lag_' in col_name:
        return 'Temporal Lags'
    elif '_roll_' in col_name or '_ma_' in col_name or '_ema_' in col_name:
        return 'Rolling Stats'
    elif '_return_' in col_name or '_vol_' in col_name or '_bb_' in col_name:
        return 'Returns & Volatility'
    elif col_name.startswith('temp_') or col_name.startswith('prec_') or col_name in ['oni', 'et0_global', 'gdd_30d_global', 'heat_stress_days_30d', 'prec_deficit_30d']:
        return 'Climate'
    elif col_name.startswith('cftc_'):
        return 'CFTC Sentiment'
    elif col_name.startswith('gdelt_') or col_name.startswith('sentiment_') or col_name.startswith('tone_') or col_name.startswith('article_'):
        return 'GDELT Sentiment'
    elif col_name.startswith('bdi_'):
        return 'Baltic Dry Index'
    elif col_name.startswith('crop_'):
        return 'Crop Conditions'
    elif col_name.startswith('gov_stocks_'):
        return 'Government Stocks'
    else:
        return 'Base Features'

# Agrupar missing por tipo
missing_summary['feature_type'] = missing_summary.index.map(clasificar_feature)
missing_by_type = missing_summary.groupby('feature_type').agg({
    'missing_count': ['sum', 'mean', 'count'],
    'missing_pct': 'mean'
}).round(2)

print("\n" + "=" * 80)
print("MISSING VALUES POR TIPO DE FEATURE")
print("=" * 80)
print(missing_by_type)


MISSING VALUES POR TIPO DE FEATURE
                     missing_count                missing_pct
                               sum     mean count        mean
feature_type                                                 
Baltic Dry Index              3344   836.00     4        4.14
Base Features              4011793  1957.93  2049        9.71
GDELT Sentiment               8220   822.00    10        4.07
Returns & Volatility       1345633  1716.37   784        8.51
Temporal Lags                 2844   948.00     3        4.70


## 3. Estrategia de Limpieza de NaNs

**Principios:**
1. **Temporal Lags & Rolling Stats:** Forward fill (ffill) - usar valor anterior
2. **Returns & Volatility:** Median imputation (volatilidad histórica)
3. **Climate Features:** Median imputation (promedio climático)
4. **CFTC/GDELT/BDI:** Forward fill (sentiment se mantiene hasta nuevo dato)
5. **Crop/Gov Stocks:** Forward fill (datos mensuales/trimestrales)
6. **Base Features:** Ya no deberían tener missing

**Gap 2014 GDELT:**
- GDELT 1.0: 2000-2013
- GDELT 2.0: 2015-2025
- 2014: Forward fill desde 2013 (asumir sentiment estable)

**Orden de aplicación:**
1. ffill para features temporales (lags, rolling, CFTC, GDELT, BDI, Crop, Stocks)
2. Median imputation para features calculados (returns, volatility, clima)
3. Verificación final (assert no quedan NaNs)

In [25]:
 # Crear copia para limpieza
df_clean = df_features.copy()

# Registrar operaciones de limpieza
cleaning_log = {
    'timestamp': datetime.now().isoformat(),
    'input_shape': df_features.shape,
    'total_missing_before': int(df_features.isnull().sum().sum()),
    'operations': []
}

print("=" * 80)
print("INICIANDO LIMPIEZA DE NaNs")
print("=" * 80)
print(f"Missing inicial: {df_features.isnull().sum().sum():,} ({(df_features.isnull().sum().sum() / df_features.size * 100):.2f}%)")

INICIANDO LIMPIEZA DE NaNs
Missing inicial: 5,371,834 (8.20%)


### 3.1 Forward Fill para Features Temporales

In [26]:
# Identificar columnas temporales (lags, rolling, CFTC, GDELT, BDI, Crop, Stocks)
temporal_cols = [
    col for col in df_clean.columns 
    if '_lag_' in col or '_roll_' in col or '_ma_' in col or '_ema_' in col
    or col.startswith('cftc_') or col.startswith('gdelt_') or col.startswith('sentiment_')
    or col.startswith('tone_') or col.startswith('article_')
    or col.startswith('bdi_') or col.startswith('crop_') or col.startswith('gov_stocks_')
]

print(f"\n[1] Forward Fill para {len(temporal_cols)} features temporales")
print(f"    Incluye: lags, rolling, CFTC, GDELT, BDI, Crop, Gov Stocks")
missing_before_ffill = df_clean[temporal_cols].isnull().sum().sum()

# Aplicar ffill (nueva sintaxis sin warnings)
df_clean[temporal_cols] = df_clean[temporal_cols].ffill()

missing_after_ffill = df_clean[temporal_cols].isnull().sum().sum()
print(f"    Missing eliminado: {missing_before_ffill:,} → {missing_after_ffill:,} (Δ = {missing_before_ffill - missing_after_ffill:,})")

# Si quedan NaNs en features académicas (primeros días), usar bfill
if missing_after_ffill > 0:
    print(f"    Aplicando backfill para primeros días...")
    df_clean[temporal_cols] = df_clean[temporal_cols].bfill()
    missing_after_bfill = df_clean[temporal_cols].isnull().sum().sum()
    print(f"    Missing después de bfill: {missing_after_bfill:,}")

# Registrar operación
cleaning_log['operations'].append({
    'step': 1,
    'method': 'forward_fill + backfill',
    'columns_count': len(temporal_cols),
    'missing_before': int(missing_before_ffill),
    'missing_after': int(df_clean[temporal_cols].isnull().sum().sum())
})


[1] Forward Fill para 17 features temporales
    Incluye: lags, rolling, CFTC, GDELT, BDI, Crop, Gov Stocks
    Missing eliminado: 14,408 → 460 (Δ = 13,948)
    Aplicando backfill para primeros días...
    Missing después de bfill: 0


### 3.2 Median Imputation para Features Calculados

In [27]:
# Identificar columnas para median imputation (todo excepto temporales)
median_cols = [col for col in df_clean.columns if col not in temporal_cols]

# Separar columnas numéricas y categóricas
numeric_median_cols = [col for col in median_cols if df_clean[col].dtype in ['float64', 'int64']]
categorical_cols = [col for col in median_cols if df_clean[col].dtype == 'object']

print(f"\n[2] Median Imputation para {len(numeric_median_cols)} features numéricos")
missing_before_median = df_clean[numeric_median_cols].isnull().sum().sum()

# Contadores para columnas toda NaN
all_nan_cols = []

# Aplicar median imputation a columnas numéricas (sin imprimir cada warning)
for col in numeric_median_cols:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        
        # Si la columna es toda NaN (edge case), usar 0
        if pd.isna(median_val):
            median_val = 0
            all_nan_cols.append(col)
        
        df_clean[col] = df_clean[col].fillna(median_val)

missing_after_median = df_clean[numeric_median_cols].isnull().sum().sum()

# Imprimir resumen de columnas toda NaN (no cada una)
if len(all_nan_cols) > 0:
    print(f"    ⚠️ {len(all_nan_cols)} columnas eran toda NaN → imputadas con 0")
    # Solo mostrar las primeras 5 como ejemplo
    if len(all_nan_cols) <= 5:
        print(f"       {all_nan_cols}")
    else:
        print(f"       Ejemplos: {all_nan_cols[:5]} ...")

print(f"    Missing eliminado: {missing_before_median:,} → {missing_after_median:,} (Δ = {missing_before_median - missing_after_median:,})")

# Aplicar mode imputation a columnas categóricas
if len(categorical_cols) > 0:
    print(f"\n[3] Mode Imputation para {len(categorical_cols)} features categóricos")
    missing_before_cat = df_clean[categorical_cols].isnull().sum().sum()
    
    for col in categorical_cols:
        if df_clean[col].isnull().sum() > 0:
            mode_val = df_clean[col].mode()
            if len(mode_val) > 0:
                df_clean[col] = df_clean[col].fillna(mode_val[0])
            else:
                df_clean[col] = df_clean[col].fillna('Unknown')
    
    missing_after_cat = df_clean[categorical_cols].isnull().sum().sum()
    print(f"    Missing eliminado: {missing_before_cat:,} → {missing_after_cat:,} (Δ = {missing_before_cat - missing_after_cat:,})")
else:
    missing_before_cat = 0
    missing_after_cat = 0

# Registrar operación
cleaning_log['operations'].append({
    'step': 2,
    'method': 'median_imputation',
    'columns_count': len(numeric_median_cols),
    'all_nan_columns': len(all_nan_cols),
    'missing_before': int(missing_before_median),
    'missing_after': int(missing_after_median)
})

if len(categorical_cols) > 0:
    cleaning_log['operations'].append({
        'step': 3,
        'method': 'mode_imputation',
        'columns_count': len(categorical_cols),
        'missing_before': int(missing_before_cat),
        'missing_after': int(missing_after_cat)
    })


[2] Median Imputation para 3227 features numéricos


f:\miniconda\envs\ds\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
f:\miniconda\envs\ds\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
f:\miniconda\envs\ds\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
f:\miniconda\envs\ds\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
f:\miniconda\envs\ds\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
f:\miniconda\envs\ds\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1231: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out,

    ⚠️ 29 columnas eran toda NaN → imputadas con 0
       Ejemplos: ['Baltic_Dry_Index_volume_price_to_ma7', 'Heat_Stress_Days_price_to_ma7', 'Baltic_Dry_Index_volume_price_to_ma30', 'Heat_Stress_Days_price_to_ma30', 'Baltic_Dry_Index_volume_price_to_ma90'] ...
    Missing eliminado: 5,357,406 → 0 (Δ = 5,357,406)

[3] Mode Imputation para 2 features categóricos
    Missing eliminado: 20 → 0 (Δ = 20)


### 3.3 Verificación Final

In [28]:
# Verificar que no quedan NaNs
total_missing_final = df_clean.isnull().sum().sum()
print("\n" + "=" * 80)
print("VERIFICACIÓN FINAL")
print("=" * 80)
print(f"Missing después de limpieza: {total_missing_final:,}")

# Assert crítico
assert total_missing_final == 0, f"ERROR: Todavía quedan {total_missing_final:,} NaNs después de limpieza"
print("✓ Sin missing values")

# [4] Limpiar valores infinitos
df_numeric = df_clean.select_dtypes(include=[np.number])
inf_before = np.isinf(df_numeric).sum().sum()
print(f"\n[4] Limpieza de Infinitos")
print(f"    Valores infinitos encontrados: {inf_before:,}")

if inf_before > 0:
    # Reemplazar infinitos por NaN y luego median
    for col in df_numeric.columns:
        col_inf = np.isinf(df_clean[col]).sum()
        if col_inf > 0:
            # Reemplazar inf por NaN
            df_clean[col] = df_clean[col].replace([np.inf, -np.inf], np.nan)
            # Imputar con mediana
            median_val = df_clean[col].median()
            if pd.isna(median_val):
                median_val = 0
            df_clean[col] = df_clean[col].fillna(median_val)
    
    # Verificar que no quedan infinitos
    df_numeric_clean = df_clean.select_dtypes(include=[np.number])
    inf_after = np.isinf(df_numeric_clean).sum().sum()
    print(f"    Infinitos después de limpieza: {inf_after:,}")
else:
    inf_after = 0
    print("    ✓ Sin valores infinitos")

# Actualizar contadores
inf_counts = inf_after

print(f"\nShape: {df_clean.shape}")
print(f"Período: {df_clean.index.min()} → {df_clean.index.max()}")
print("\n✓ DATASET LIMPIO - Sin missing values ni infinitos")

# Actualizar log
cleaning_log['total_missing_after'] = int(total_missing_final)
cleaning_log['infinite_before'] = int(inf_before)
cleaning_log['infinite_after'] = int(inf_after)
cleaning_log['output_shape'] = list(df_clean.shape)


VERIFICACIÓN FINAL
Missing después de limpieza: 0
✓ Sin missing values

[4] Limpieza de Infinitos
    Valores infinitos encontrados: 160,882

[4] Limpieza de Infinitos
    Valores infinitos encontrados: 160,882
    Infinitos después de limpieza: 0

Shape: (20173, 3246)
Período: 2000-01-03 00:00:00 → 2025-11-10 00:00:00

✓ DATASET LIMPIO - Sin missing values ni infinitos
    Infinitos después de limpieza: 0

Shape: (20173, 3246)
Período: 2000-01-03 00:00:00 → 2025-11-10 00:00:00

✓ DATASET LIMPIO - Sin missing values ni infinitos


## 4. Validación de Calidad del Dataset Final

### 4.1 Verificar Continuidad Temporal

In [29]:
# Verificar que no hay gaps en el índice temporal
date_range = pd.date_range(start=df_clean.index.min(), end=df_clean.index.max(), freq='D')
missing_dates = date_range.difference(df_clean.index)

print("=" * 80)
print("VALIDACIÓN DE CONTINUIDAD TEMPORAL")
print("=" * 80)
print(f"Fecha inicial: {df_clean.index.min()}")
print(f"Fecha final: {df_clean.index.max()}")
print(f"Días esperados: {len(date_range):,}")
print(f"Días en dataset: {len(df_clean):,}")
print(f"Fechas faltantes: {len(missing_dates)}")

if len(missing_dates) > 0:
    print(f"\nWARNING: {len(missing_dates)} fechas faltantes:")
    print(missing_dates[:10])  # Mostrar primeras 10
else:
    print("\n✓ Serie temporal continua sin gaps")

VALIDACIÓN DE CONTINUIDAD TEMPORAL
Fecha inicial: 2000-01-03 00:00:00
Fecha final: 2025-11-10 00:00:00
Días esperados: 9,444
Días en dataset: 20,173
Fechas faltantes: 2713

DatetimeIndex(['2000-01-08', '2000-01-09', '2000-01-15', '2000-01-16', '2000-01-22', '2000-01-23', '2000-01-29', '2000-01-30', '2000-02-05', '2000-02-06'], dtype='datetime64[ns]', freq=None)


### 4.2 Verificar Ranges de Variables

In [30]:
# Verificar que no hay valores infinitos o extremos anómalos
print("\n" + "=" * 80)
print("VALIDACIÓN DE RANGES")
print("=" * 80)

# Seleccionar solo columnas numéricas
df_numeric = df_clean.select_dtypes(include=[np.number])

# Infinitos
inf_counts = np.isinf(df_numeric).sum().sum()
print(f"Valores infinitos: {inf_counts}")

if inf_counts > 0:
    inf_cols = df_numeric.columns[np.isinf(df_numeric).any()].tolist()
    print(f"    Columnas con infinitos: {inf_cols[:10]}")  # Primeras 10
else:
    print("    ✓ Sin valores infinitos")


VALIDACIÓN DE RANGES
Valores infinitos: 0
    ✓ Sin valores infinitos
Valores infinitos: 0
    ✓ Sin valores infinitos


### 4.3 Verificar Correlación con Targets

In [31]:
# Identificar columnas target (precios base de commodities)
target_candidates = [col for col in df_clean.columns if col in ['soy', 'corn', 'wheat', 'coffee', 'sugar']]

if len(target_candidates) > 0:
    print("\n" + "=" * 80)
    print("CORRELACIÓN CON TARGETS (Precios Base)")
    print("=" * 80)
    
    for target in target_candidates:
        # Calcular correlación con todas las features
        corr_with_target = df_clean.corr()[target].sort_values(ascending=False)
        
        print(f"\n{target.upper()} - Top 10 features más correlacionadas:")
        print(corr_with_target.head(10))
else:
    print("\nWARNING: No se encontraron targets en el dataset")

## 5. Guardar Dataset Final para Modelado

In [32]:
# Guardar dataset limpio
output_path = PROCESSED_DIR / 'features_final_modeling.csv'
df_clean.to_csv(output_path)

print("=" * 80)
print("GUARDANDO DATASET FINAL")
print("=" * 80)
print(f"✓ Dataset guardado: {output_path}")
print(f"  Shape: {df_clean.shape}")
print(f"  Size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")
print(f"  Missing values: 0 (verificado)")

GUARDANDO DATASET FINAL
✓ Dataset guardado: C:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal\data\processed\features_final_modeling.csv
  Shape: (20173, 3246)
  Size: 851.78 MB
  Missing values: 0 (verificado)


## 6. Guardar Metadata Completa

In [33]:
# Guardar metadata como JSON
metadata = {
    'creation_date': datetime.now().isoformat(),
    'notebook': '2.7-final-dataset-preparation.ipynb',
    'description': 'Dataset final limpio para modelado con sentiment consolidado GDELT 1.0+2.0 (1979-2025)',
    
    'sources': [
        'features_step4_climate.csv',
        'cftc/cftc_features_2000_2025.csv',
        'sentiment/sentiment_features_1979_2025.csv',
        'bdi/bdi_features.csv',
        'supply_demand/crop_conditions_all_features.csv',
        'supply_demand/government_stocks_ers_all_features.csv'
    ],
    'cleaning_applied': True,
    
    # Dimensiones
    'shape': {
        'rows': df_clean.shape[0],
        'columns': df_clean.shape[1]
    },
    
    # Período temporal
    'temporal_range': {
        'start': df_clean.index.min().isoformat(),
        'end': df_clean.index.max().isoformat(),
        'days': len(df_clean),
        'missing_dates': len(missing_dates)
    },
    
    # Calidad de datos
    'data_quality': {
        'missing_values': 0,
        'infinite_values': int(inf_counts),
        'total_cells': int(df_clean.size)
    },
    
    # Composición de features (nombres consistentes)
    'feature_composition': {
        'targets': 3,
        'temporal_lags': len([c for c in df_clean.columns if '_lag' in c]),
        'rolling_stats': len([c for c in df_clean.columns if '_ma' in c or 'std' in c]),
        'returns_volatility': len([c for c in df_clean.columns if 'return' in c or 'vol_ratio' in c or '_bb_' in c]),
        'climate': len([c for c in df_clean.columns if 'Temp' in c or 'Precip' in c or 'GDD' in c or 'ET0' in c or 'RH_' in c or 'Wind' in c or 'Solar' in c or 'ONI' in c or 'Heat_Stress' in c]),
        'cftc_sentiment': len([c for c in df_clean.columns if 'managed' in c or 'producer' in c or 'swap' in c or 'other_' in c]),
        'gdelt_sentiment': len([c for c in df_clean.columns if 'tone_' in c or 'article_' in c]),
        'bdi': len([c for c in df_clean.columns if 'Baltic_Dry' in c or 'bdi_' in c]),
        'crop_conditions': len([c for c in df_clean.columns if 'crop_' in c or 'gov_stocks' in c]),
        'gov_stocks': len([c for c in df_clean.columns if 'gov_stocks' in c]),
        'psd_fundamentals': len([c for c in df_clean.columns if 'psd_' in c]),
        'technical_indicators': len([c for c in df_clean.columns if 'is_outlier' in c or 'is_planting' in c or 'is_harvest' in c or 'is_month' in c or 'is_quarter' in c or 'season' in c])
    }
}

# Guardar JSON
metadata_path = PROCESSED_DIR / 'metadata_final_dataset.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

logger.info(f"Metadata guardada: {metadata_path}")
print(f"\n✅ Metadata guardada: {metadata_path.name}")
print(f"   Features por categoría:")
for cat, count in metadata['feature_composition'].items():
    print(f"   - {cat}: {count}")

2025-12-07 13:41:08 - config - INFO - Metadata guardada: C:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal\data\processed\metadata_final_dataset.json



✅ Metadata guardada: metadata_final_dataset.json
   Features por categoría:
   - targets: 3
   - temporal_lags: 323
   - rolling_stats: 885
   - returns_volatility: 1569
   - climate: 331
   - cftc_sentiment: 15
   - gdelt_sentiment: 10
   - bdi: 71
   - crop_conditions: 24
   - gov_stocks: 9
   - psd_fundamentals: 640
   - technical_indicators: 314


## Resumen Final

In [34]:
print("\n" + "=" * 80)
print("RESUMEN - DATASET FINAL PARA MODELADO")
print("=" * 80)

print("\n✓ DATASET LIMPIO Y VALIDADO")
print(f"  - Shape: {df_clean.shape[0]:,} días × {df_clean.shape[1]:,} features")
print(f"  - Período: {df_clean.index.min().date()} → {df_clean.index.max().date()}")
print(f"  - Missing values: 0")
print(f"  - Infinite values: {inf_counts}")

print("\n✓ COMPOSICIÓN DE FEATURES:")
for cat, count in metadata['feature_composition'].items():
    print(f"  - {cat}: {count}")

print("\n✓ LIMPIEZA APLICADA:")
print(f"  - Forward fill + backfill: {cleaning_log['operations'][0]['missing_before'] - cleaning_log['operations'][0]['missing_after']:,} NaNs eliminados")
print(f"  - Median imputation: {cleaning_log['operations'][1]['missing_before'] - cleaning_log['operations'][1]['missing_after']:,} NaNs eliminados")
print(f"  - Infinitos limpiados: {cleaning_log.get('infinite_before', 0):,}")

print("\n✓ ARCHIVOS GENERADOS:")
print(f"  - {output_path.name} ({output_path.stat().st_size / 1024 / 1024:.1f} MB)")
print(f"  - {metadata_path.name}")

print("\n" + "=" * 80)
print("DATASET LISTO PARA NOTEBOOKS DE MODELADO (4.x)")
print("=" * 80)
print("\n⚠️ IMPORTANTE: Este dataset incluye SENTIMENT (GDELT tone_*)")
print("   Verificar que 4.1-feature-selection use este archivo actualizado")


RESUMEN - DATASET FINAL PARA MODELADO

✓ DATASET LIMPIO Y VALIDADO
  - Shape: 20,173 días × 3,246 features
  - Período: 2000-01-03 → 2025-11-10
  - Missing values: 0
  - Infinite values: 0

✓ COMPOSICIÓN DE FEATURES:
  - targets: 3
  - temporal_lags: 323
  - rolling_stats: 885
  - returns_volatility: 1569
  - climate: 331
  - cftc_sentiment: 15
  - gdelt_sentiment: 10
  - bdi: 71
  - crop_conditions: 24
  - gov_stocks: 9
  - psd_fundamentals: 640
  - technical_indicators: 314

✓ LIMPIEZA APLICADA:
  - Forward fill + backfill: 14,408 NaNs eliminados
  - Median imputation: 5,357,406 NaNs eliminados
  - Infinitos limpiados: 160,882

✓ ARCHIVOS GENERADOS:
  - features_final_modeling.csv (851.8 MB)
  - metadata_final_dataset.json

DATASET LISTO PARA NOTEBOOKS DE MODELADO (4.x)

⚠️ IMPORTANTE: Este dataset incluye SENTIMENT (GDELT tone_*)
   Verificar que 4.1-feature-selection use este archivo actualizado
